### 0. Colab Setup: Install Dependencies
Run the cell below to install the necessary libraries for the tutorial.

In [ ]:
# This is required if you are running this notebook on Google Colab
!pip install -q qdrant-client sentence-transformers transformers opendatasets

# Simple RAG System: WE Telecom Customer Agent

###  Problem Statement
Customers of WE Telecom often experience the hassle of manually searching for information about pricing plans, internet packages, or company directories. Finding the right information quickly can be time-consuming and frustrating.

###  Objective
This notebook builds an informational **Customer Agent** using Retrieval-Augmented Generation (RAG). By combining a vector database (Qdrant) and a large language model (BART), we will create a smart assistant capable of instantly and accurately answering customer queries based on WE Telecom's official documentation.

---
*This notebook merges the database indexing and querying into a single, simple procedural flow without relying on the uploaded `.md` documents.*

### High-Level RAG Architecture

**1. Data Ingestion (Indexing) Phase**
This phase is all about preparing your data so the AI can quickly search and understand it.

*   **Knowledge / Info:** These are your raw source documents (in this case, the WE Telecom markdown files). They contain the facts you want your AI to know.
*   **Embeddings / Vectors:** Computers don't understand words natively, so we use an Encoder model (BERT/MiniLM) to translate sentences into long arrays of numbers (vectors). Text with similar meanings will have numbers that are mathematically close to each other.
*   **Qdrant Vector DB:** A vector database like Qdrant stores those arrays of numbers and calculates the "distance" between them to find text that is *semantically related*.

**`📄 Knowledge` ➔ `🧮 Embeddings` ➔ `🗄️ Qdrant Vector DB`**

**2. Query & Generation (Retrieval) Phase**
This is what happens live when a user asks a question.

*   **Question:** The user types a query in plain English.
*   **Embedding / Vector:** We pass the user's question through the *exact same* Encoder model (BERT) to turn that question into a vector.
*   **Search in Qdrant:** We throw the question's vector into Qdrant and find the closest matching documents.
*   **Retrieve Relevant Docs:** Qdrant returns the top matches (e.g., the top 2 closest markdown files). We now have the factual context needed to answer the question.
*   **BART Generates Answer:** We are using BART, which is a generative LLM (Decoder). We take the user's original question and the text we just retrieved from Qdrant, and we feed them both to BART. BART reads the context and writes a natural, conversational answer based *only* on those facts.

**`❓ Question` ➔ `🧮 Embedding` ➔ `🔍 Search Qdrant` ➔ `📄 Retrieve Docs` ➔ `🤖 BART Generates Answer`**

### Saving Embedings (Knowledge) Pipeline

**`1️⃣ Load Encoder Model` ➔ `2️⃣ Connect to Qdrant Cloud` ➔ `3️⃣ Read Markdown Files` ➔ `4️⃣ Encode to Vectors` ➔ `5️⃣ Push to Database`**

*Model Description: [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)*

In [ ]:
import os
import warnings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
from transformers import BartTokenizer, BartForConditionalGeneration

warnings.filterwarnings('ignore')

### 1. Load Models
Load our encoder (BERT) to convert text to vectors and decoder (BART) to generate answers.

In [ ]:
print('Loading Models...')
encoder_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
encoder_dim = encoder_model.get_sentence_embedding_dimension()

decoder_tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
decoder_model = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa')
print('Models loaded successfully!')

### 2. Connect to Qdrant Database
Connect to your free Qdrant Cloud cluster and initialize our collection. 

**How to get your credentials:**
1. Go to [Qdrant Cloud](https://cloud.qdrant.io/) and create a free account.
2. Create a new "Free Tier" cluster.
3. Once provisioned, click on your cluster to open its details.
4. Copy the **Cluster URL** (it should look like `https://<your-cluster>.<region>.cloud.qdrant.io:6333`) and generate/copy an **API Key**. 
   *(Note: Do NOT copy the dashboard URL from your browser address bar).*

In [ ]:
import getpass

print('Connecting to Qdrant Cloud...')

# Securely prompt for Qdrant Cloud credentials
QDRANT_URL = getpass.getpass('Enter your Qdrant Cloud Cluster URL: ').strip()
QDRANT_API_KEY = getpass.getpass('Enter your Qdrant Cloud API Key: ').strip()

# Quick validation to help avoid common URL errors
if 'cloud.qdrant.io/clusters' in QDRANT_URL:
    print("\nWARNING: It looks like you copied the Dashboard URL instead of the Cluster URL.")
    print("Please use the Cluster URL which looks like: https://<id>.<region>.cloud.qdrant.io:6333\n")

if not QDRANT_URL.startswith('http'):
    QDRANT_URL = 'https://' + QDRANT_URL
    
# Qdrant Cloud requires port 6333 to be explicitly stated in the URL
if 'cloud.qdrant.io' in QDRANT_URL and not QDRANT_URL.endswith(':6333'):
    QDRANT_URL += ':6333'

try:
    qdrant = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY
    )

    collection_name = 'we_telecom_knowledge'

    if qdrant.collection_exists(collection_name):
        print(f"Collection '{collection_name}' already exists. Recreating it...")
        qdrant.delete_collection(collection_name)

    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=encoder_dim, distance=Distance.COSINE),
    )
    print('Qdrant collection ready!')
    
except Exception as e:
    print(f"\nConnection Error: Make sure your Cluster URL and API Key are exactly as they appear in the Qdrant Cloud console.")
    print("Error Details:", str(e))

### 3. Download and Index Documents
We will download the dataset from Kaggle to access the markdown files. We will then read the files, encode them, and store them into Qdrant.

### Download Dataset

We will use the `opendatasets` library to download the dataset directly from Kaggle. You will need your Kaggle username and Kaggle API key (which can be created in your Kaggle Account Settings).
[Kaggle Dataset Link](https://www.kaggle.com/datasets/mahmoudramadan025/we-telecom-scraped-data)

**`📄 Knowledge` ➔ `🧮 Embeddings` ➔ `🗄️ Qdrant Vector DB`**

In [ ]:
import os

import opendatasets as od
# Download dataset from Kaggle (It will ask for Kaggle username and key)
dataset_url = 'https://www.kaggle.com/datasets/mahmoudramadan025/we-telecom-scraped-data'
od.download(dataset_url)
dataset_path = 'we-telecom-scraped-data'


print('Reading Markdown files and indexing...')
documents = []
points = []

# Search for markdown files in the dataset folder
for root, dirs, files in os.walk(dataset_path):
    # Prefer files from the English folder if present
    if 'WE_Telecom_en' in root or dataset_path == root:
        for file_name in files:
            if file_name.endswith('.md'):
                with open(os.path.join(root, file_name), 'r', encoding='utf-8') as f:
                    text = f.read()
                    # Prevent duplicates if dataset_path == root also matches the subdirectory later
                    if not any(doc['title'] == file_name for doc in documents):
                        documents.append({'title': file_name, 'content': text})

for i, doc in enumerate(documents):
    vector = encoder_model.encode(doc['content']).tolist()
    point = PointStruct(
        id=i, 
        vector=vector, 
        payload={'title': doc['title'], 'content': doc['content']}
    )
    points.append(point)

if points:
    qdrant.upsert(collection_name=collection_name, points=points)
    print(f'Successfully indexed {len(points)} documents!')
else:
    print("No documents found to index.")

### 4. Create the Question-Answering Function
This simple function retrieves the top matching files and passes them alongside the question to BART for the answer.

**`❓ Question` ➔ `🧮 Embedding` ➔ `🔍 Search Qdrant` ➔ `📄 Retrieve Docs` ➔ `🤖 BART Generates Answer`**

In [ ]:
def ask_question(question, top_k=2):
    print(f'\n--- Question: {question} ---')
    
    # 1. RETRIEVE context from database
    query_vector = encoder_model.encode(question).tolist()
    search_results = qdrant.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    ).points
    
    print('\n[Retrieved Documents]:')
    context_texts = []
    for hit in search_results:
        print(f" - Match: {hit.payload['title']} (Score: {hit.score:.2f})")
        context_texts.append(hit.payload['content'])
        
    combined_context = ' '.join([text.replace('\n', ' ').strip() for text in context_texts])
    
    # 2. GENERATE answer with BART
    input_text = f"question: {question} context: {combined_context}"
    inputs = decoder_tokenizer(
        [input_text], 
        max_length=1024, 
        truncation=True, 
        return_tensors='pt'
    )
    
    summary_ids = decoder_model.generate(
        inputs['input_ids'], 
        max_length=200,             # Maximum allowed length for the output answer
        min_length=10,              # Minimum required length for the output answer
        num_beams=5,                # Evaluates top 5 sentence paths in parallel (Beam Search)
        no_repeat_ngram_size=3,     # Forbids the model from repeating the same 3-word string 
        length_penalty=1.5,         # Mathematically rewards generating longer, descriptive sentences
        early_stopping=True         # Stops generation early once all beam paths reach an end token
    )
    
    answer = decoder_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f'\n[BART Answer]:\n{answer}\n')
    
    return answer

### 5. Test the RAG system
Run questions to test the generated RAG system pipeline directly.

In [ ]:
ask_question("What are the benefits of WE Internet Space Super?")

---

## 🛠️ Task 1: Build Your Own Knowledge Base

Now that you've seen how the WE Telecom customer agent works, it's your turn to build a custom RAG system!

**Your Mission:**
1. **Scrape:** Choose a specific website (e.g., your university's site, a favorite blog, or a wiki) and scrape the content from at least **5 different pages**.
2. **Save:** Save the scraped content into individual `.md` (Markdown) files.
3. **Upload:** Create a new folder on your Google Drive (e.g., `My_Custom_Knowledge`) and upload your `.md` files there.
4. **Index and Query:** Write code to read these new files, index them into the Qdrant database, and test a few questions against your new knowledge base!

**⚠️ Important Hints:**
*   **Update the Collection Name:** Don't use `we_telecom_knowledge` for your new data. Change the `collection_name` variable in your code to something like `my_custom_knowledge` so you don't overwrite the previous data!
*   **Update the Drive Path:** Make sure you update the `docs_path` variable in the indexing step to point to your new Google Drive folder (e.g., `'/content/drive/MyDrive/My_Custom_Knowledge'`).